# K-Nearest Neighbors (KNN) classification on Titanic Dataset

## 1. Import Libraries
In this step, we import the necessary libraries: pandas/numpy for data manipulation, matplotlib/seaborn for plotting, and scikit-learn modules for modeling, metrics, and preprocessing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

## 2. Load the Dataset
We load the raw Titanic dataset CSV file into a Pandas DataFrame.

In [ ]:
dataset = "../../Datasets/titanic.csv"

df = pd.read_csv(dataset)

df

## 3. Preview the Data
We inspect the first 5 and last 5 rows of the dataset to get a quick visual overview of the features.

In [ ]:
print(df.head())
print()
print(df.tail())

## 4. Dataset Shape
Check the total number of rows and columns (features) in the dataset.

In [ ]:
df.shape

## 5. Dataset Summary
Check the data types of each column and identify columns with missing (null) values.

In [ ]:
df.info()

## 6. Descriptive Statistics
Summarize the central tendency, dispersion, and shape of the dataset's numerical distribution.

In [ ]:
df.describe()

## 7. Count Missing Values
Identify exactly how many missing values are present in each column to decide on our cleaning strategy.

In [ ]:
df.isnull().sum()

## 8. Check for Duplicate Rows
Verify if there are any duplicate records in the dataset to avoid training bias.

In [ ]:
df.duplicated().sum()

## 9. Drop Irrelevant Columns
We drop features like `PassengerId`, `Name`, `Ticket`, and `Cabin` because they contain unique/text values that do not provide generalizable patterns for survival prediction. We use `errors='ignore'` so the cell can be re-run safely without raising KeyErrors.

In [ ]:
df.drop(["Cabin", "Name", "Ticket", "PassengerId"], axis=1, inplace=True, errors="ignore")
df

## 10. Verify Remaining Columns
Check the columns left after dropping the irrelevant features.

In [ ]:
df.columns

## 11. Handle Missing Values in Embarked
Since only 2 rows are missing the `Embarked` value, we can safely drop those 2 rows from the dataset.

In [ ]:
df.dropna(subset=["Embarked"], inplace=True)

## 12. Impute Missing Age & Encode Categorical Features
* **Age Imputation:** We fill the missing values in the `Age` column with its median value, which is robust against outliers. We assign it back directly to avoid Copy-on-Write warnings.
* **Categorical Encoding:** We use `LabelEncoder` to convert the string categories of `Sex` and `Embarked` into numerical values so that the KNN algorithm can calculate distances.

In [ ]:
df["Age"] = df["Age"].fillna(df["Age"].median())

encoder = LabelEncoder()

df["Sex"] = encoder.fit_transform(df["Sex"])
df["Embarked"] = encoder.fit_transform(df["Embarked"])

df

## 13. Feature Engineering: Family Metrics
* **FamilySize:** We combine siblings/spouses (`SibSp`) and parents/children (`Parch`) to calculate the total family size on board.
* **IsAlone:** A binary feature (0 or 1) indicating if the passenger traveled alone.

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

## 14. Separate Features and Target
We separate the dataset into the features matrix `X` and the target vector `y` (survival status).

In [ ]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

## 15. Train-Test Split
We split our data into a Training set (80%) to train the model and a Test set (20%) to evaluate it on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 16. Feature Scaling (Standardization)
Since KNN relies on Euclidean or Manhattan distance calculations, features with larger scales (like `Fare` or `Age`) would dominate features with smaller scales (like `Sex` or `Pclass`). We standardize them so they all have a mean of 0 and a standard deviation of 1. Crucially, we fit the scaler on the training set only to avoid data leakage.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 17. Train the KNN Model
We initialize the K-Nearest Neighbors Classifier (with $K=5$) and train it on the scaled training data.

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5)

knn_model.fit(X_train_scaled, y_train)

## 18. Predict on Test Set
We use our trained model to make survival predictions on the scaled test data.

In [ ]:
y_pred = knn_model.predict(X_test_scaled)

y_pred

## 19. Evaluate Performance
We calculate the accuracy, precision, recall, and F1-score of our predictions on the test set, and visualize the Confusion Matrix.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Survived (0)', 'Survived (1)'],
            yticklabels=['Not Survived (0)', 'Survived (1)'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("Classification Report\n")
print(classification_report(y_test, y_pred))

## 20. Feature Scaling for Cross-Validation
To perform cross-validation without using a pipeline, we standardize the entire features set `X` first. (Note: In production environments, using a `Pipeline` or manual fold-scaling loop is preferred to avoid data leakage).

In [ ]:
scale = StandardScaler()

X_scaled = scale.fit_transform(X)

## 21. K-Fold Cross-Validation
We evaluate the model's stability and average performance using 5-fold cross-validation on the scaled features, calculating the mean accuracy and standard deviation.

In [ ]:
scores = cross_val_score(knn_model, X_scaled, y, cv=5)
print(f"All 5 fold accuracies: {scores}")
print(f"Mean Accuracy: {scores.mean():.4f}")
print(f"Standard Deviation: {scores.std():.4f}")

### 22. Summary and Key Findings

This notebook demonstrated the implementation of a K-Nearest Neighbors (KNN) classifier to predict survival outcomes on the Titanic dataset, detailing the baseline data preparation, model evaluation, and cross-validation performance.

### Data Analysis Key Findings
* **Model Classification Performance (Test Set)**: The base KNN model (with $K=5$ and Euclidean distance) achieved an overall test accuracy of **79.78%**, with a precision of **73.24%**, recall of **75.36%**, and an F1-score of **74.29%** on the test set.
* **Confusion Matrix Details (Base Model)**:
  * **True Negatives**: 90 passengers correctly predicted to not survive.
  * **True Positives**: 52 passengers correctly predicted to survive.
  * **False Positives**: 19 passengers predicted to survive but actually did not.
  * **False Negatives**: 17 passengers predicted to not survive but actually did.
* **Cross-Validation Baseline**:
  * Running a 5-fold cross-validation on the base scaled features yielded a baseline mean accuracy of **80.09%** with a standard deviation of **1.95%**.

### Insights or Next Steps
* **Hyperparameter Tuning**: The baseline KNN model uses default parameters ($K=5$, Euclidean distance). Performing a grid search to optimize $K$, distance metrics (e.g., Manhattan distance), and weight functions would be the logical next step to improve model accuracy.
* **Feature Optimization**: Exploring additional feature transformations (like log-scaling for the highly skewed `Fare` column and one-hot encoding for multi-class categorical features like `Embarked`) could help reduce distance bias and improve predictions.